<a href="https://colab.research.google.com/github/kawastony/Quadratic-Mechanism-Lens/blob/main/TIFA_figures_paper1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# Figure 1 for TIFA Paper 1
# w(z): TIFA trajectory vs CPL vs DESI constraints
# Save as: figure1_paper1.pdf
# Run: python figure1_paper1.py
# Requires: numpy, matplotlib, scipy
# ============================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms
from scipy.integrate import solve_ivp

# ── Matplotlib style ─────────────────────────────────────────
plt.rcParams.update({
    'font.family':        'serif',
    'font.size':          12,
    'axes.labelsize':     13,
    'axes.titlesize':     13,
    'legend.fontsize':    11,
    'xtick.labelsize':    11,
    'ytick.labelsize':    11,
    'text.usetex':        False,
    'figure.dpi':         150,
    'axes.linewidth':     1.2,
    'xtick.direction':    'in',
    'ytick.direction':    'in',
    'xtick.top':          True,
    'ytick.right':        True,
})

# ── TIFA parameters ──────────────────────────────────────────
# Natural units: M_Pl = 1
feff   = 0.305          # decay constant in M_Pl
Omega_DE = 0.69
Omega_m  = 0.31
Omega_r  = 9.0e-5
h        = 0.674
H0       = 1.0          # working units H0 = 1

rho_DE_0 = 3.0 * H0**2 * Omega_DE   # ~ 3 * 0.69
Lambda4  = 0.1 * rho_DE_0
Lambda   = Lambda4**0.25
phi_i    = 0.377 * np.pi * feff

# ── PNGB potential ───────────────────────────────────────────
def V(phi):
    return Lambda4 * (1.0 + np.cos(phi / feff))

def dV(phi):
    return -Lambda4 / feff * np.sin(phi / feff)

# ── Hubble in units of H0 ────────────────────────────────────
def H_background(a, rho_phi):
    rho_m = Omega_m * a**(-3)
    rho_r = Omega_r * a**(-4)
    rho_tot = rho_m + rho_r + rho_phi
    return np.sqrt(rho_tot / 3.0)

# ── ODE system: variables [phi, dphi_da, a] ──────────────────
# Use scale factor a as independent variable
def odes(lna, y):
    phi, phidot_H = y   # phi, and dot_phi / H
    a = np.exp(lna)

    rho_m   = Omega_m * a**(-3)
    rho_r   = Omega_r * a**(-4)
    Vphi    = V(phi)
    dVphi   = dV(phi)

    # phidot = phidot_H * H
    # rho_phi = 0.5 * phidot^2 + V
    # We need H self-consistently
    # Let x = phidot_H, then phidot = x*H
    # rho_phi = 0.5 x^2 H^2 + V

    # Solve for H:
    # 3H^2 = rho_m + rho_r + 0.5 x^2 H^2 + V
    # H^2 (3 - 0.5 x^2) = rho_m + rho_r + V
    denom = 3.0 - 0.5 * phidot_H**2
    if denom <= 0:
        denom = 1e-10
    H2 = (rho_m + rho_r + Vphi) / denom
    H  = np.sqrt(max(H2, 1e-30))

    phidot = phidot_H * H

    rho_phi = 0.5 * phidot**2 + Vphi
    rho_tot = rho_m + rho_r + rho_phi
    p_phi   = 0.5 * phidot**2 - Vphi

    # Hdot / H^2 = -3/2 - (rho_r/2 + 3*p_phi/2) / (3H^2)
    # More precisely:
    # dH/dt = -1/(2 M_Pl^2) (rho_tot + p_tot)
    # p_tot = p_r + p_phi = rho_r/3 + p_phi
    p_tot   = rho_r / 3.0 + p_phi
    Hdot_over_H2 = -0.5 * (rho_tot + p_tot) / H2

    # dphi/dlna = phidot / H / a * a = phidot / H
    dphi_dlna = phidot / H

    # d(phidot/H)/dlna:
    # phidot_H = phidot / H
    # d phidot_H / dlna = (ddot_phi - phidot * Hdot) / H^2
    # ddot_phi = -3H phidot - dV
    ddotphi = -3.0 * H * phidot - dVphi
    d_phidotH_dlna = (ddotphi / H - phidot_H * Hdot_over_H2 * H) / H

    return [dphi_dlna, d_phidotH_dlna]

# ── Initial conditions ───────────────────────────────────────
a_i   = 1.0 / (1.0 + 4.0)   # z_i = 4
lna_i = np.log(a_i)
lna_f = 0.0                   # z = 0, a = 1

# Slow-roll initial phidot/H
H_i_approx = np.sqrt(
    (Omega_m * a_i**(-3) + Omega_r * a_i**(-4) + V(phi_i)) / 3.0
)
phidot_i = -dV(phi_i) / (3.0 * H_i_approx)
phidot_H_i = phidot_i / H_i_approx

y0 = [phi_i, phidot_H_i]

# ── Integrate ────────────────────────────────────────────────
lna_span = (lna_i, lna_f)
lna_eval = np.linspace(lna_i, lna_f, 500)

sol = solve_ivp(
    odes, lna_span, y0,
    t_eval=lna_eval,
    method='DOP853',
    rtol=1e-9, atol=1e-11
)

lna_arr = sol.t
a_arr   = np.exp(lna_arr)
z_arr   = 1.0 / a_arr - 1.0
phi_arr = sol.y[0]
phiH_arr = sol.y[1]

# ── Compute w(z) ─────────────────────────────────────────────
w_arr = np.zeros(len(z_arr))
for i in range(len(z_arr)):
    phi = phi_arr[i]
    xH  = phiH_arr[i]
    a   = a_arr[i]
    Vphi = V(phi)
    rho_m = Omega_m * a**(-3)
    rho_r = Omega_r * a**(-4)
    denom = 3.0 - 0.5 * xH**2
    H2 = (rho_m + rho_r + Vphi) / max(denom, 1e-10)
    phidot2 = xH**2 * H2
    rho_phi = 0.5 * phidot2 + Vphi
    p_phi   = 0.5 * phidot2 - Vphi
    w_arr[i] = p_phi / max(rho_phi, 1e-30)

# Sort by redshift ascending for plotting
idx = np.argsort(z_arr)
z_plot = z_arr[idx]
w_plot = w_arr[idx]

# ── CPL parametrisation ──────────────────────────────────────
w0_desi = -0.826
wa_desi = -0.264

def w_cpl(z, w0, wa):
    return w0 + wa * z / (1.0 + z)

z_cpl = np.linspace(0, 4, 300)
w_cpl_arr = w_cpl(z_cpl, w0_desi, wa_desi)

# ── DESI 1-sigma and 2-sigma bands ───────────────────────────
# Propagate CPL uncertainty to w(z)
# sigma_w0 = 0.085, sigma_wa = 0.290, correlation ~ -0.9
sigma_w0 = 0.085
sigma_wa = 0.290
rho_corr = -0.90   # approximate from DESI paper

def w_band(z, w0, wa, dw0, dwa, rho, nsigma):
    # Variance of w(z) = (dw/dw0)^2 sigma_w0^2
    #                  + (dw/dwa)^2 sigma_wa^2
    #                  + 2 rho sigma_w0 sigma_wa (dw/dw0)(dw/dwa)
    f = z / (1.0 + z)
    var = (dw0)**2 + (f * dwa)**2 + 2 * rho * dw0 * (f * dwa)
    std = np.sqrt(np.abs(var))
    w_center = w0 + wa * f
    return w_center - nsigma * std, w_center + nsigma * std

w_lo1, w_hi1 = w_band(
    z_cpl, w0_desi, wa_desi,
    sigma_w0, sigma_wa, rho_corr, 1
)
w_lo2, w_hi2 = w_band(
    z_cpl, w0_desi, wa_desi,
    sigma_w0, sigma_wa, rho_corr, 2
)

# ── Plot ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7.5, 5.5))

# DESI 2-sigma band
ax.fill_between(
    z_cpl, w_lo2, w_hi2,
    color='#AED6F1', alpha=0.5,
    label=r'DESI $2\sigma$ band'
)

# DESI 1-sigma band
ax.fill_between(
    z_cpl, w_lo1, w_hi1,
    color='#2E86C1', alpha=0.4,
    label=r'DESI $1\sigma$ band'
)

# CPL best fit
ax.plot(
    z_cpl, w_cpl_arr,
    color='#1A5276', linewidth=1.8,
    linestyle='--',
    label=r'CPL best fit ($w_0=-0.826,\ w_a=-0.264$)'
)

# TIFA trajectory
ax.plot(
    z_plot, w_plot,
    color='#C0392B', linewidth=2.5,
    label=r'TIFA ($f_{\rm eff}=0.305\,M_{\rm Pl}$)'
)

# Lambda CDM reference
ax.axhline(
    -1.0, color='gray', linewidth=1.2,
    linestyle=':', label=r'$\Lambda$CDM ($w=-1$)'
)

# Axis formatting
ax.set_xlabel(r'Redshift $z$', labelpad=8)
ax.set_ylabel(r'Equation of state $w(z)$', labelpad=8)
ax.set_xlim(0, 4)
ax.set_ylim(-1.15, -0.70)
ax.set_title(
    r'TIFA Dark-Energy Equation of State vs DESI Constraints',
    pad=10
)
ax.legend(loc='lower right', framealpha=0.9)
ax.grid(True, alpha=0.25, linestyle='--')

# Annotate key values
ax.annotate(
    r'$w_0 = -0.826$' + '\n' + r'$w_a = -0.264$',
    xy=(0.05, 0.12),
    xycoords='axes fraction',
    fontsize=10.5,
    bbox=dict(boxstyle='round,pad=0.3',
              facecolor='white', alpha=0.8)
)

ax.annotate(
    r'$f_{\rm eff} = 0.305\,M_{\rm Pl}$',
    xy=(0.60, 0.88),
    xycoords='axes fraction',
    fontsize=10.5,
    color='#C0392B',
    bbox=dict(boxstyle='round,pad=0.3',
              facecolor='white', alpha=0.8)
)

plt.tight_layout()
plt.savefig('figure1_paper1.pdf', bbox_inches='tight')
plt.savefig('figure1_paper1.png', bbox_inches='tight', dpi=200)
print("Saved: figure1_paper1.pdf and figure1_paper1.png")

Saved: figure1_paper1.pdf and figure1_paper1.png
